In [ ]:

from data_loader import (
    RemoteDataLoader,
    SqliteDataLoader,
    JsonFileDataLoader,
    DataFrameDataLoader,
    create_loader
)
from generation import load_unii_data, enhance_unii_data, find_unii_data_for_substance
import pandas as pd

json_loader = JsonFileDataLoader(file_path='docs/data.json')
substance_df = pd.DataFrame(json_loader.load())

substance_df["DISPLAY_NAME"] = substance_df["Name"].str.upper()

unii_df = enhance_unii_data(load_unii_data())


In [ ]:
substance_unii_df = substance_df.merge(unii_df, how="left", left_on="DISPLAY_NAME", right_on="DISPLAY_NAME")

In [ ]:
from pathlib import Path
import requests
import json

drugsfda_path = Path("./.cache/drugsfda")
drugsfda_path.mkdir(exist_ok=True, parents=True)
def get_or_query_fda_data(row):
    unii = row.UNII
    if not isinstance(unii, str) or row.DRUGSFDA_DATA == {}:
        return {}
    filepath = drugsfda_path / unii
    try:
        if not filepath.exists():
            url = f'https://api.fda.gov/drug/drugsfda.json?search=products.unii={unii}&limit=99'
            r = requests.get(url)
            r.raise_for_status()
            data = r.json()
            with filepath.open('w') as f:
                json.dump(data, f)
            return data
        else:
            return json.load(filepath.open())
    except requests.models.HTTPError as e:
        print(e)
        return {}

substance_unii_df["DRUGSFDA_DATA"] = substance_unii_df.apply(get_or_query_fda_data, axis=1)

In [ ]:
pd.json_normalize(substance_unii_df["DRUGSFDA_DATA"], )

# openFDA  

https://open.fda.gov/data/downloads/  

https://open.fda.gov/data/datadictionary


In [ ]:
import requests
r = requests.get(
    f"https://api.fda.gov/drug/drugsfda.json", 
    params={
        "search": 'products.dosage_form:"LOTION"',
        "limit":1
        }
    )
print(r.url)
r
# search=products.dosage_form:%22LOTION%22&limit=1

In [ ]:
r = requests.get(
    f"https://api.fda.gov/drug/drugsfda.json", 
    params={ # caffeine
        "search": 'products.unii=3G6A5W338E',
        "limit":99
        }
    )
print(r.url)
r

In [ ]:
r = requests.get(
    f"https://api.fda.gov/drug/drugsfda.json", 
    params={
        "search": 'products.unii=7ZW49N180B',
        "limit":100
        }
    )
print(r.url)
r

In [ ]:
r.url
# https://api.fda.gov/drug/drugsfda.json?search=products.unii=3G6A5W338E&limit=99

In [ ]:
d = r.json()
d

In [ ]:
# Thank you for your interest in the Computational Toxicology and Exposure (CT) APls. Your CT API key is 
# https://www.epa.gov/comptox-tools/computational-toxicology-and-exposure-apis-about
# https://github.com/USEPA/ctx-python
import ctxpy as ctx

chem = ctx.Chemical()

In [ ]:
chem.search(by='equals', query='toluene')

In [ ]:
# https://comptox.epa.gov/ctx-api/docs/exposure.html
exposure = ctx.Exposure() 

In [ ]:
from urllib.parse import quote as quote_url
# from urllib.parse import 
# chem.search(by='equals', query=["1","1-phenylcyclohexyl","pyrrolidine"])
# https://gooosetavo.github.io/dod-prohibited/substances/17-dihydroexemestane: DTXSID00464522
# https://precision.fda.gov/uniisearch/srs/unii/U35ZNO9DJH

In [ ]:
dtxsid = "DTXSID0020232" # caffeine 

product_info = exposure.search_cpdat(vocab_name='puc', dtxsid=dtxsid)
# "puc" is product information, and 
functional_use = exposure.search_cpdat(vocab_name='fc', dtxsid=dtxsid)
#  "fc" is reported functional use, 
predicted_functional_use = exposure.search_cpdat(vocab_name='qsur', dtxsid=dtxsid)
# "qsur" is predicted functional use,
list_presence = exposure.search_cpdat(vocab_name='lpk', dtxsid=dtxsid)
#"lpk" is chemical list presence information.

In [ ]:
product_info.head(5)

In [ ]:
functional_use.head(5)

In [ ]:
list_presence.head(5)

In [ ]:
exposure.search_qsurs